In [1]:
# Parameters
BATCH_MODE = "true"

# Fournisseur réellement exécuté par ce notebook.
DEFAULT_PROVIDER = "openai"
OPENAI_MODEL = "gpt-5.6-luna"

print(f"Paramètres chargés : fournisseur={DEFAULT_PROVIDER}, modèle={OPENAI_MODEL}")

Paramètres chargés : fournisseur=openai, modèle=gpt-5.6-luna


# SC-11-LLM-Assisted-Python - Développement Smart Contracts Assisté par LLM

[<< Précédent : Account Abstraction](SC-10-Account-Abstraction-Python.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Foundry Testing >>](../03-Foundry-Testing/SC-12-Foundry-Testing-Python.ipynb)

***

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Utiliser un **LLM réellement invoqué** pour générer du code Solidity
2. Appliquer des **patterns de prompting** efficaces pour les smart contracts
3. Exploiter un LLM pour la **détection de vulnérabilités**
4. Générer automatiquement la **documentation NatSpec**
5. Intégrer l'API **OpenAI** dans un workflow de développement
6. Compiler et tester sur une EVM locale le Solidity généré avant de le considérer comme utilisable
7. Situer un **LLM local** (Qwen) comme alternative optionnelle

### Prérequis

- Notebooks SC-1 à SC-4 (fondements Solidity)
- Clé `OPENAI_API_KEY` dans un fichier `.env` gitignored
- Python 3.10+ avec `openai`, `python-dotenv`, `py-solc-x`, `web3`, `eth-tester`, `py-evm` et `requests`

### Durée estimée : 45 minutes

Plan du notebook : la section 1 configure le client LLM et le compilateur ; la section 2 définit les templates de prompts ; la section 3 génère, compile et teste un contrat sur une EVM PyEVM locale ; la section 4 audite un contrat volontairement vulnérable ; la section 5 ajoute la documentation NatSpec avec recompilation ; la section 6 enchaîne le pipeline complet ; la section 7 présente l'extension locale Ollama. Chaque démonstration est réellement exécutée et sa sortie est conservée dans le carnet.

***

## Introduction

Les **Large Language Models** ont revolutionne le développement de smart contracts. Ils peuvent :

- **Generer du code** a partir de specifications en langage naturel
- **Auditer** des contrats pour detecter des vulnerabilites
- **Documenter** automatiquement avec des commentaires NatSpec
- **Expliquer** des concepts complexes Solidity

### Pourquoi utiliser les LLMs pour Solidity ?

| Avantage | Description |
|----------|-------------|
| **Rapidite** | Prototypage en minutes, pas en heures |
| **Apprentissage** | Explications contextuelles du code genere |
| **Securite** | Detection precoce de patterns vulnerables |
| **Documentation** | NatSpec genere automatiquement |

### Limitations importantes

> **Attention** : Les LLMs peuvent generer du code contenant des bugs ou vulnerabilites.
> Toujours auditer, tester et verifier formellement le code genere avant deploiement en production.

Dans ce carnet, chaque démonstration suit le même contrat de preuve : le LLM produit une proposition, puis un outil déterministe la valide — `solc` pour la compilabilité, PyEVM pour les comportements testés, la recompilation pour le NatSpec. Une cellule dont la sortie ne porte pas cette validation n'est pas présentée comme réussie.

## 1. Configuration et Imports

In [2]:
# Imports standards
import os
import re
import sys
from dataclasses import dataclass
from typing import Any, Dict

import requests
from dotenv import load_dotenv
from eth_tester import EthereumTester, PyEVMBackend
from eth_tester.exceptions import TransactionFailed
from openai import OpenAI
from solcx import compile_source, get_installed_solc_versions, install_solc
from solcx.exceptions import SolcError
from web3 import Web3
from web3.providers.eth_tester import EthereumTesterProvider

sys.path.insert(0, os.path.abspath(".."))
from web3_tx import transact_and_wait

load_dotenv()
print(
    "Imports chargés : openai, python-dotenv, py-solc-x, web3, "
    "eth-tester, py-evm, requests"
)

Imports chargés : openai, python-dotenv, py-solc-x, web3, eth-tester, py-evm, requests


Configuration du client OpenAI et du compilateur Solidity. Le fournisseur est choisi explicitement dans la cellule de paramètres ; la présence opportuniste d'une autre clé ne change jamais le chemin exécuté.

La sortie de la cellule précédente affiche `Imports chargés : openai, python-dotenv, py-solc-x, web3, eth-tester, py-evm, requests` : ce sont exactement les outils invoqués ensuite — `openai` pour les appels LLM, `solcx` pour compiler avec `solc 0.8.20`, `web3` avec `eth-tester`/`py-evm` pour déployer et tester sur une EVM locale, et `python-dotenv` pour charger la clé depuis le fichier `.env`.

In [3]:
@dataclass
class LLMConfig:
    """Configuration du fournisseur réellement exécuté."""

    provider: str = DEFAULT_PROVIDER
    model: str = OPENAI_MODEL
    max_completion_tokens: int = 4096
    timeout_seconds: float = 90.0


config = LLMConfig()
if config.provider != "openai":
    raise ValueError(f"Fournisseur non pris en charge dans ce run : {config.provider}")

openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY manque. Ajoutez-la à un fichier .env gitignored avant d'exécuter SC-11."
    )

llm_client = OpenAI(api_key=openai_api_key, timeout=config.timeout_seconds)
print(f"Client {config.provider} initialisé avec le modèle {config.model}")
print("Clé API chargée depuis l'environnement (valeur non affichée)")

Client openai initialisé avec le modèle gpt-5.6-luna
Clé API chargée depuis l'environnement (valeur non affichée)


### Interprétation : prérequis d'exécution

Le notebook exige `OPENAI_API_KEY` avant toute démonstration : si la clé manque, l'exécution s'arrête avec une erreur explicite et ne fabrique aucun résultat de substitution.

**Configuration locale** :
1. Créez un fichier `.env` dans le répertoire SmartContracts ;
2. ajoutez `OPENAI_API_KEY=<votre-cle>` ;
3. gardez ce fichier hors Git — `.env` est déjà ignoré par le dépôt.

La cellule confirme seulement que la clé est chargée ; elle n'en affiche jamais la valeur ni un préfixe.

La sortie de la cellule précédente affiche `Client openai initialisé avec le modèle gpt-5.6-luna` puis `Clé API chargée depuis l'environnement (valeur non affichée)` : le client est prêt avec le modèle épinglé dans la cellule de paramètres, et la clé reste secrète — jamais imprimée, ni en clair ni tronquée.

## 2. Prompt Patterns pour Solidity

Le prompting efficace pour Solidity necessite structure et precision. Voici les templates de base.

Un prompt efficace pour Solidity suit toujours la même anatomie, visible dans les templates de la cellule suivante : un rôle précis, des contraintes exécutables (version du compilateur, contrat autonome, noms de fonctions attendus par le harnais) et un format de sortie strict (« retourne uniquement un bloc de code compilable ») qui rend la réponse directement exploitable par `extract_solidity_code`.

In [4]:
# Templates de prompts pour Solidity
SOLIDITY_PROMPTS = {
    "contract_generation": """Tu es un expert en développement de smart contracts Solidity.

Génère un contrat Solidity complet pour le cas d'usage suivant :
{description}

Contraintes :
- Version Solidity : ^0.8.20
- Contrat autonome : aucun import ni dépendance externe
- Constructeur sans paramètre : le contrat est déployé sans argument
- Nommer exactement les fonctions appelées par le harnais : `store(uint256)` et `retrieve()`
- Ajouter les commentaires NatSpec complets
- Implémenter les contrôles de sécurité adaptés
- Respecter exactement les fonctionnalités demandées
- Solc 0.8.20 : pas d'assembly inline accédant à une variable immutable (ex. `owner.slot` non supporté) — écrire les transferts de propriété en Solidity simple
- Utiliser uniquement des caractères ASCII dans les chaînes littérales exécutées par le contrat

Retourne uniquement un bloc de code Solidity compilable, sans explication supplémentaire.
""",
    "audit": """Tu es un auditeur de sécurité de smart contracts.

Analyse ce contrat Solidity pour les vulnérabilités potentielles :
```solidity
{code}
```

Identifie :
1. les vulnérabilités connues ;
2. les problèmes de gas ;
3. les risques de centralisation ;
4. les bonnes pratiques non respectées.

Pour chaque finding, utilise exactement une ligne `SEVERITY: CRITICAL`, `SEVERITY: HIGH`,
`SEVERITY: MEDIUM`, `SEVERITY: LOW` ou `SEVERITY: INFO`, puis donne la fonction concernée,
l'explication et une correction. N'ajoute aucun préfixe Markdown sur la ligne `SEVERITY`.
Si aucun finding n'existe, retourne exactement `NO_FINDINGS` sur une ligne isolée.
Ne prétends pas avoir exécuté un outil d'analyse statique.
""",
    "natspec": """Ajoute une documentation NatSpec complète à ce contrat Solidity sans modifier son comportement :
```solidity
{code}
```

Inclus uniquement les tags valides pour chaque déclaration : @notice et @dev selon le besoin,
@title et @author uniquement au niveau du contrat (solc les rejette sur une fonction ou un événement),
@param exactement une fois par paramètre, et @return uniquement pour une fonction qui déclare une valeur
de retour. Ne modifie aucune signature, instruction ni chaîne littérale exécutée par le contrat.
Retourne uniquement le contrat complet dans un bloc de code Solidity compilable.
""",
    "explain": """Explique ce code Solidity de manière pédagogique :
```solidity
{code}
```

Présente son objectif, son état, ses fonctions, ses contrôles de sécurité et ses limites.
""",
}

print("Templates de prompts définis :", ", ".join(SOLIDITY_PROMPTS))

Templates de prompts définis : contract_generation, audit, natspec, explain


### Lecture de la sortie : les quatre templates de prompts

La sortie affiche `Templates de prompts définis : contract_generation, audit, natspec, explain` : quatre gabarits réutilisés par toutes les cellules suivantes. Chacun fixe un rôle (« expert en développement de smart contracts »), des contraintes exécutables (version `^0.8.20`, contrat autonome sans import, noms de fonctions imposés `store(uint256)` et `retrieve()`) et un format de sortie strict (uniquement un bloc de code Solidity compilable), ce qui rend la réponse directement exploitable par `extract_solidity_code`.

### 2.1 Bonnes pratiques de prompting

| Pratique | Exemple | Impact |
|----------|---------|--------|
| **Spécifier la version** | « Solidity ^0.8.20 » | Code compatible |
| **Borner les dépendances** | « Contrat autonome, sans import » | Compilation reproductible |
| **Demander NatSpec** | « Ajoute les commentaires NatSpec » | Documentation automatique |
| **Contraintes de sécurité** | « Applique Checks-Effects-Interactions » | Contrôles explicites |
| **Format de sortie** | « Retourne uniquement le code » | Extraction fiable |

## 3. Génération et compilation de contrats via API

Le chemin principal invoque réellement OpenAI, extrait le bloc Solidity, puis le compile avec `py-solc-x`. Une réponse textuelle non compilable est donc rejetée au lieu d'être présentée comme un contrat utilisable.

Trois échecs sont propagés explicitement plutôt que masqués : une réponse LLM vide lève `RuntimeError`, un artefact solc vide lève `RuntimeError`, et un test EVM qui contredit le comportement attendu lève `AssertionError`. Aucun chemin ne fabrique de résultat de substitution.

In [5]:
SOLC_VERSION = "0.8.20"


def extract_solidity_code(response_text: str) -> str:
    """Extrait un bloc Solidity d'une réponse LLM."""
    solidity_blocks = re.findall(
        r"```solidity\s*\n(.*?)```", response_text, re.DOTALL | re.IGNORECASE
    )
    if solidity_blocks:
        return solidity_blocks[0].strip()

    generic_blocks = re.findall(r"```\s*\n(.*?)```", response_text, re.DOTALL)
    if generic_blocks:
        return generic_blocks[0].strip()
    return response_text.strip()


def call_llm(prompt: str) -> str:
    """Invoque le fournisseur configuré et retourne sa réponse textuelle."""
    response = llm_client.chat.completions.create(
        model=config.model,
        max_completion_tokens=config.max_completion_tokens,  # gpt-5.6-luna : max_completion_tokens requis, temperature non supportee
        messages=[{"role": "user", "content": prompt}],
    )
    content = response.choices[0].message.content
    if not content:
        raise RuntimeError("OpenAI a retourné une réponse vide.")
    return content


def compile_contract(solidity_code: str) -> dict[str, Any]:
    """Compile le Solidity généré et retourne l'ABI et le bytecode."""
    installed = {str(version) for version in get_installed_solc_versions()}
    if SOLC_VERSION not in installed:
        install_solc(SOLC_VERSION)

    artifacts = compile_source(
        solidity_code,
        output_values=["abi", "bin"],
        solc_version=SOLC_VERSION,
    )
    if not artifacts:
        raise RuntimeError("solc n'a produit aucun artefact.")
    contract_id, artifact = next(iter(artifacts.items()))
    return {
        "contract_id": contract_id,
        "abi": artifact["abi"],
        "bytecode": artifact["bin"],
    }


def test_simple_storage(compilation: dict[str, Any]) -> dict[str, Any]:
    """Déploie et teste SimpleStorage sur une EVM PyEVM locale."""
    tester = EthereumTester(backend=PyEVMBackend())
    web3 = Web3(EthereumTesterProvider(tester))
    owner, outsider = web3.eth.accounts[:2]

    contract_factory = web3.eth.contract(
        abi=compilation["abi"], bytecode=compilation["bytecode"]
    )
    deployment_receipt = transact_and_wait(
        web3, contract_factory.constructor(), owner
    )
    contract = web3.eth.contract(
        address=deployment_receipt.contractAddress, abi=compilation["abi"]
    )

    initial_value = contract.functions.retrieve().call()
    update_receipt = transact_and_wait(web3, contract.functions.store(42), owner)
    stored_value = contract.functions.retrieve().call()

    outsider_rejected = False
    try:
        # Sous EthereumTesterProvider, un revert est levé dès .transact(),
        # donc par l'organe lui-même : le try/except reste le garde attendu.
        transact_and_wait(web3, contract.functions.store(7), outsider)
    except TransactionFailed:
        outsider_rejected = True

    if initial_value != 0 or stored_value != 42 or not outsider_rejected:
        raise AssertionError(
            "Le test EVM de SimpleStorage n'a pas vérifié le comportement attendu."
        )

    return {
        "backend": type(tester.backend).__name__,
        "deployment_status": deployment_receipt.status,
        "update_status": update_receipt.status,
        "initial_value": initial_value,
        "stored_value": stored_value,
        "outsider_rejected": outsider_rejected,
    }


def generate_contract(description: str) -> tuple[str, str]:
    """Génère un contrat Solidity avec le fournisseur réellement configuré."""
    prompt = SOLIDITY_PROMPTS["contract_generation"].format(description=description)
    full_response = call_llm(prompt)
    solidity_code = extract_solidity_code(full_response)
    return solidity_code, full_response


print("Fonctions LLM, compilation et test EVM définies")

Fonctions LLM, compilation et test EVM définies


Génération d'un contrat Solidity simple à partir d'une description en langage naturel via l'API LLM.

La sortie affiche `Fonctions LLM, compilation et test EVM définies` : le pipeline de la section 3 tient en quatre fonctions — `call_llm` (appel au fournisseur configuré), `extract_solidity_code` (extraction du bloc Solidity balisé), `compile_contract` (compilation `solc 0.8.20`, avec installation si le binaire manque) et `test_simple_storage` (déploiement PyEVM, `store(42)`, contrôle d'accès).

In [6]:
# Exemple : générer, compiler et tester un contrat simple
description = """
Un contrat SimpleStorage autonome, sans import externe, qui permet :
- stocker une valeur uint256 ;
- récupérer la valeur stockée ;
- autoriser uniquement le propriétaire à modifier la valeur ;
- émettre un événement ValueChanged quand la valeur change.
"""

print("Génération d'un contrat de stockage avec OpenAI...")
code, response = generate_contract(description)
compiled = compile_contract(code)
behavior = test_simple_storage(compiled)

print("CONTRAT GÉNÉRÉ :")
print("=" * 60)
print(code)
print("\nCOMPILATION SOLC : SUCCÈS")
print(f"Contrat : {compiled['contract_id']}")
print(f"Entrées ABI : {len(compiled['abi'])}")
print(f"Bytecode : {len(compiled['bytecode']) // 2} octets")
print("\nTEST EVM : SUCCÈS")
print(f"Backend : {behavior['backend']}")
print(f"Déploiement : statut {behavior['deployment_status']}")
print(f"Écriture propriétaire : statut {behavior['update_status']}")
print(f"Lecture initiale : {behavior['initial_value']}")
print(f"Lecture après store(42) : {behavior['stored_value']}")
print(f"Écriture non-propriétaire rejetée : {behavior['outsider_rejected']}")

Génération d'un contrat de stockage avec OpenAI...


CONTRAT GÉNÉRÉ :
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title SimpleStorage
/// @notice Stores a single uint256 value that can only be modified by the owner.
/// @dev The contract is autonomous and has no external dependencies.
contract SimpleStorage {
    /// @notice The address authorized to modify the stored value.
    address public owner;

    /// @notice The value currently stored by the contract.
    uint256 private storedValue;

    /// @notice Emitted when the stored value changes.
    /// @param newValue The new value stored by the contract.
    event ValueChanged(uint256 indexed newValue);

    /// @notice Emitted when ownership is transferred.
    /// @param previousOwner The address of the previous owner.
    /// @param newOwner The address of the new owner.
    event OwnershipTransferred(
        address indexed previousOwner,
        address indexed newOwner
    );

    /// @notice Restricts access to the current owner.
    modifier onlyOwner() {


### Interprétation : génération vérifiée

La cellule précédente ne s'arrête pas à la réponse du LLM : elle passe le code extrait à `solc 0.8.20`, puis déploie l'artefact sur une EVM locale fournie par `PyEVMBackend`. Le test vérifie l'état initial, l'écriture propriétaire, la lecture de la valeur `42` et le rejet d'une écriture provenant d'un autre compte.

Les nombres d'entrées ABI et d'octets de bytecode proviennent de l'artefact compilé dans ce run ; les statuts de transaction et valeurs lues proviennent de l'exécution EVM. Cette preuve comportementale bornée ne garantit pas la sûreté générale du contrat. L'audit LLM qui suit reste une aide à la revue, pas un substitut aux tests Foundry plus complets, au fuzzing et à l'expertise humaine.

Sortie citée telle quelle : `Contrat : <stdin>:SimpleStorage`, `Entrées ABI : 6`, puis le test EVM `Backend : PyEVMBackend`, `Déploiement : statut 1`, `Écriture propriétaire : statut 1`, `Lecture initiale : 0`, `Lecture après store(42) : 42`, `Écriture non-propriétaire rejetée : True`. Le contrat généré respecte donc le cahier des charges : valeur initiale nulle, écriture réservée au propriétaire, lecture de 42 après `store(42)`.

## 4. Audit de Securite avec LLM

Utilisons les LLMs pour detecter les vulnerabilites dans les smart contracts.

Le prompt d'audit impose un format exploitable : une ligne `SEVERITY: CRITICAL|HIGH|MEDIUM|LOW|INFO` par finding (fonction concernée, explication, correction), ou exactement `NO_FINDINGS` si le contrat est propre — et il interdit de prétendre avoir exécuté un outil d'analyse statique. `normalize_audit_report`, définie dans la cellule suivante, reformate le rapport et rejette toute réponse qui ne respecte ni le format SEVERITY ni `NO_FINDINGS`.

In [7]:
def normalize_audit_report(report: str) -> str:
    """Normalise les lignes de sévérité et rejette un rapport non structuré."""
    normalized_lines = []
    severities = []

    for line in report.splitlines():
        severity_match = re.search(
            r"(?i)\bSEVERITY\s*:\s*(CRITICAL|HIGH|MEDIUM|LOW|INFO)\b",
            line,
        )
        if severity_match:
            severity = severity_match.group(1).upper()
            severities.append(severity)
            normalized_lines.append(f"SEVERITY: {severity}")
            continue

        if re.fullmatch(r"\s*NO_FINDINGS\s*", line, re.IGNORECASE):
            normalized_lines.append("NO_FINDINGS")
            continue

        normalized_lines.append(line.rstrip())

    if severities:
        normalized_lines = [line for line in normalized_lines if line != "NO_FINDINGS"]
    elif "NO_FINDINGS" not in normalized_lines:
        raise RuntimeError(
            "Le rapport d'audit ne contient ni ligne SEVERITY structurée ni NO_FINDINGS."
        )

    return "\n".join(normalized_lines).strip()


def audit_contract(code: str) -> str:
    """Demande au fournisseur configuré d'auditer un contrat Solidity."""
    prompt = SOLIDITY_PROMPTS["audit"].format(code=code)
    raw_report = call_llm(prompt)
    return normalize_audit_report(raw_report)


print("Fonctions d'audit et de normalisation définies")

Fonctions d'audit et de normalisation définies


Exemple d'audit automatique d'un contrat volontairement vulnérable. Le rapport est produit par un appel OpenAI réel ; il reste une aide à la revue et non une analyse statique formelle.

La sortie affiche `Fonctions d'audit et de normalisation définies` : `audit_contract` envoie le contrat au fournisseur via le template `audit`, et `normalize_audit_report` garantit que le rapport rendu contient au moins une ligne `SEVERITY: ...` structurée ou la ligne `NO_FINDINGS` — un rapport libre non structuré lève `RuntimeError` au lieu de passer silencieusement.

In [8]:
# Exemple: Auditer un contrat vulnerables
vulnerable_contract = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.0;

contract VulnerableVault {
    mapping(address => uint256) public balances;
    
    function deposit() external payable {
        balances[msg.sender] += msg.value;
    }
    
    function withdraw() external {
        uint256 amount = balances[msg.sender];
        // VULNERABILITE: appel externe avant mise a jour
        (bool success, ) = msg.sender.call{value: amount}("");
        require(success, "Transfer failed");
        balances[msg.sender] = 0;
    }
}
"""

print("=== AUDIT DE SECURITE ===")
print("Contrat analyse: VulnerableVault")
print("\n" + "="*60)

audit_result = audit_contract(vulnerable_contract)
print(audit_result)

=== AUDIT DE SECURITE ===
Contrat analyse: VulnerableVault



SEVERITY: CRITICAL
Fonction concernée : `withdraw()`. Le contrat effectue un appel externe vers `msg.sender` avant de remettre `balances[msg.sender]` à zéro. Un contrat malveillant peut donc réappeler `withdraw()` depuis sa fonction `receive()` ou `fallback()` et retirer plusieurs fois le même solde avant la mise à jour de l’état. Correction : appliquer le pattern Checks-Effects-Interactions en mettant le solde à zéro avant l’appel, et ajouter une protection `ReentrancyGuard`.

SEVERITY: LOW
Fonction concernée : `withdraw()`. L’appel `msg.sender.call{value: amount}("")` transfère tout le gas restant et dépend du comportement arbitraire du destinataire. Un destinataire peut consommer excessivement du gas ou revert, provoquant l’échec du retrait concerné. Correction : mettre l’état à jour avant l’appel, envisager un mécanisme de retrait contrôlé ou un système de paiement différé, et éviter les appels externes inutiles lorsque `amount == 0`.

SEVERITY: LOW
Fonction concernée : `withdraw()

### Interpretation : Audit de VulnerableVault

Le contrat `VulnerableVault` contient une **reentrancy classique**, la vulnerabilite la plus citee dans les hacks DeFi.

**Sequence d'attaque** :

1. L'attaquant depose 1 ETH via `deposit()`
2. Il appelle `withdraw()` — le contrat envoie 1 ETH via `msg.sender.call{value: amount}("")`
3. Le fallback de l'attaquant **reappelle** `withdraw()` avant que la remise a zero du solde ne s'execute
4. Le solde est encore a 1 ETH → retrait d'un second ETH
5. Repetition jusqu'a drainage complet du contrat

**Pattern correctif (Checks-Effects-Interactions)** :

```solidity
function withdraw() external {
    uint256 amount = balances[msg.sender];  // Checks
    balances[msg.sender] = 0;               // Effects (AVANT l'appel externe)
    (bool success, ) = msg.sender.call{value: amount}("");  // Interactions
    require(success, "Transfer failed");
}
```

La règle est simple : toute mise a jour de l'etat (**Effects**) doit preceder tout appel externe (**Interactions**). La variable `amount` est lue avant la mise a zero (**Checks**), garantissant que le reappel trouve un solde a zero.
Alternative plus robuste : le modifier `ReentrancyGuard` d'OpenZeppelin centralise cette protection.

Le rapport commité de la cellule précédente commence par `SEVERITY: CRITICAL` sur `withdraw()` (réentrance : appel externe avant remise à zéro du solde), suivi de `SEVERITY: MEDIUM` (fonds bloqués si le destinataire refuse le transfert), `SEVERITY: LOW` (appel externe inutile quand le solde est nul) et deux `SEVERITY: INFO` (absence d'événements `Deposit`/`Withdraw`, absence de risque de centralisation). La normalisation a réduit chaque ligne de sévérité à sa forme canonique `SEVERITY: <niveau>`.

## 5. Generation de Documentation NatSpec

La documentation NatSpec (Ethereum Natural Language Specification) est essentielle pour les smart contracts.

Règles de validité NatSpec reprises par le template : `@title` et `@author` uniquement au niveau du contrat (solc les rejette sur une fonction ou un événement), `@param` exactement une fois par paramètre, `@return` uniquement quand la fonction déclare une valeur de retour. `generate_natspec` recompile le contrat documenté et boucle (au plus 2 tentatives) tant que `solc` rejette les commentaires.

In [9]:
def generate_natspec(code: str, max_attempts: int = 2) -> str:
    """Génère du NatSpec et exige une recompilation avant de retourner."""
    prompt = SOLIDITY_PROMPTS["natspec"].format(code=code)
    documented_response = call_llm(prompt)

    for attempt in range(max_attempts):
        documented_code = extract_solidity_code(documented_response)
        try:
            compile_contract(documented_code)
            return documented_response
        except SolcError as error:
            if attempt == max_attempts - 1:
                raise
            documented_response = call_llm(
                f"""Corrige uniquement les commentaires NatSpec invalides du contrat ci-dessous.
Rappel : @title et @author ne sont valides qu'au niveau du contrat, jamais sur une fonction.
Ne modifie aucune signature, instruction ni chaîne littérale exécutée.
Retourne uniquement le contrat Solidity complet dans un bloc de code compilable.

Diagnostic solc :
{error.message}

Contrat documenté :
```solidity
{documented_code}
```
""",
            )

    raise RuntimeError("La génération NatSpec n'a produit aucun contrat compilable.")


print("Fonction generate_natspec définie avec validation solc bornée")

Fonction generate_natspec définie avec validation solc bornée


Exemple de documentation NatSpec produite par OpenAI puis recompilée avec `solc` pour vérifier que l'ajout de commentaires n'a pas altéré le contrat.

La sortie affiche `Fonction generate_natspec définie avec validation solc bornée` : la fonction exige que le contrat documenté soit recompilable avant de retourner la réponse, avec au plus `max_attempts = 2` appels — la boucle de correction ne sollicite à nouveau le fournisseur que si `solc` a réellement rejeté les commentaires NatSpec.

In [10]:
# Exemple : ajouter NatSpec à un contrat sans documentation
undocumented_contract = """
pragma solidity ^0.8.20;

contract Token {
    string public name = "MyToken";
    string public symbol = "MTK";
    uint8 public decimals = 18;
    uint256 public totalSupply;
    mapping(address => uint256) public balanceOf;

    event Transfer(address indexed from, address indexed to, uint256 value);

    constructor(uint256 _initialSupply) {
        totalSupply = _initialSupply * 10 ** uint256(decimals);
        balanceOf[msg.sender] = totalSupply;
    }

    function transfer(address _to, uint256 _value) public returns (bool success) {
        require(balanceOf[msg.sender] >= _value, "Insufficient balance");
        balanceOf[msg.sender] -= _value;
        balanceOf[_to] += _value;
        emit Transfer(msg.sender, _to, _value);
        return true;
    }
}
"""

print("=== GÉNÉRATION NATSPEC ===")
documented_response = generate_natspec(undocumented_contract)
documented_code = extract_solidity_code(documented_response)
documented_compilation = compile_contract(documented_code)

print(documented_code)
print("\nRECOMPILATION SOLC : SUCCÈS")
print(f"Contrat : {documented_compilation['contract_id']}")
print(f"Bytecode : {len(documented_compilation['bytecode']) // 2} octets")

=== GÉNÉRATION NATSPEC ===


pragma solidity ^0.8.20;

/// @title MyToken
/// @author Token contract author
/// @notice Contrat de jeton fongible simple avec une offre initiale attribuée au déployeur.
/// @dev Les montants sont exprimés dans la plus petite unité du jeton, avec 18 décimales.
contract Token {
    /// @notice Nom lisible du jeton.
    string public name = "MyToken";

    /// @notice Symbole du jeton.
    string public symbol = "MTK";

    /// @notice Nombre de décimales utilisées par le jeton.
    uint8 public decimals = 18;

    /// @notice Offre totale de jetons en circulation.
    uint256 public totalSupply;

    /// @notice Solde de jetons détenu par chaque adresse.
    mapping(address => uint256) public balanceOf;

    /// @notice Émis lorsqu'un transfert de jetons est effectué.
    /// @param from Adresse depuis laquelle les jetons sont transférés.
    /// @param to Adresse vers laquelle les jetons sont transférés.
    /// @param value Nombre de jetons transférés.
    event Transfer(address ind

### Interprétation : balises NatSpec

La documentation NatSpec generee inclut les tags standards :

| Tag | Usage |
|-----|-------|
| `@title` | Nom du contrat |
| `@author` | Auteur du code |
| `@notice` | Description utilisateur |
| `@dev` | Details techniques |
| `@param` | Description des paramètres |
| `@return` | Description de la valeur de retour |
| `@event` | Description des événements |

La fin de la sortie affiche `RECOMPILATION SOLC : SUCCÈS`, `Contrat : <stdin>:Token`, `Bytecode : 3843 octets` : le contrat documenté (chaque déclaration porte désormais des balises `@notice`/`@dev`/`@param`/`@return` valides) recompile sous `solc 0.8.20`, ce qui prouve que l'ajout de commentaires n'a modifié ni signature ni comportement.

## 6. Pipeline LLM vérifiable

`SolidityLLMAssistant` applique le même fournisseur explicite aux trois étapes : génération, audit et documentation. La génération est en outre compilée par `solc` avant la poursuite du pipeline. Aucun choix ne dépend de la présence opportuniste d'une autre clé et aucun fallback ne fabrique une réponse terminale.

La classe `SolidityLLMAssistant`, définie dans la cellule suivante, refuse tout fournisseur autre que celui configuré (`ValueError` sinon) et enchaîne quatre opérations sur le même modèle : `generate` (génération + compilation), `audit`, `document` (NatSpec + recompilation) et `full_workflow`, qui les combine.

In [11]:
class SolidityLLMAssistant:
    """Assistant OpenAI pour générer, compiler, auditer et documenter Solidity."""

    def __init__(self, provider: str = DEFAULT_PROVIDER):
        if provider != config.provider:
            raise ValueError(f"Fournisseur non configuré : {provider}")
        self.provider = provider
        self.model = config.model

    def generate(self, description: str) -> Dict[str, Any]:
        """Génère et compile un contrat complet."""
        code, response = generate_contract(description)
        compilation = compile_contract(code)
        return {
            "code": code,
            "full_response": response,
            "compilation": compilation,
            "model": self.model,
            "provider": self.provider,
        }

    def audit(self, code: str) -> Dict[str, Any]:
        """Audite un contrat pour les vulnérabilités."""
        return {
            "report": audit_contract(code),
            "model": self.model,
            "provider": self.provider,
        }

    def document(self, code: str) -> Dict[str, Any]:
        """Ajoute la documentation NatSpec."""
        documented = generate_natspec(code)
        documented_code = extract_solidity_code(documented)
        compile_contract(documented_code)
        return {
            "documented_code": documented_code,
            "full_response": documented,
            "model": self.model,
            "provider": self.provider,
        }

    def full_workflow(self, description: str) -> Dict[str, Any]:
        """Exécute génération -> compilation -> audit -> documentation."""
        generated = self.generate(description)
        audited = self.audit(generated["code"])
        documented = self.document(generated["code"])
        return {
            "original_code": generated["code"],
            "documented_code": documented["documented_code"],
            "audit_report": audited["report"],
            "compilation": generated["compilation"],
            "model": self.model,
            "provider": self.provider,
        }


assistant = SolidityLLMAssistant()
print(f"Assistant initialisé avec {assistant.provider} ({assistant.model})")
print("Pipeline actif : génération -> compilation -> audit -> documentation")

Assistant initialisé avec openai (gpt-5.6-luna)
Pipeline actif : génération -> compilation -> audit -> documentation


Démonstration du workflow complet : génération, audit et documentation d'un contrat Solidity via l'assistant LLM.

La sortie affiche `Assistant initialisé avec openai (gpt-5.6-luna)` et `Pipeline actif : génération -> compilation -> audit -> documentation` : l'assistant est instancié avec le fournisseur épinglé en cellule de paramètres, et le pipeline annoncé est celui que la démonstration suivante exécute étape par étape.

In [12]:
# Démonstration du workflow complet
print("=== WORKFLOW COMPLET LLM ===")

result = assistant.full_workflow("""
Un contrat Counter autonome, sans import externe, qui permet :
- incrémenter un compteur ;
- décrémenter le compteur sans passer sous zéro ;
- lire la valeur actuelle ;
- autoriser uniquement le propriétaire à incrémenter ou décrémenter.
""")

print("CODE GÉNÉRÉ :")
print(result["original_code"])
print("\nAUDIT LLM :")
print(result["audit_report"])
print("\nPREUVE DE COMPILATION :")
print(f"Contrat : {result['compilation']['contract_id']}")
print(f"Bytecode : {len(result['compilation']['bytecode']) // 2} octets")

=== WORKFLOW COMPLET LLM ===


CODE GÉNÉRÉ :
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title Counter
/// @notice Autonomous counter controlled exclusively by its owner.
/// @dev The contract has no external dependencies and uses Solidity 0.8.20
///      arithmetic checks to prevent overflow and underflow.
contract Counter {
    /// @notice Address authorized to modify the counter.
    address public immutable owner;

    /// @notice Current counter value.
    uint256 private _counter;

    /// @notice Emitted when the counter value changes.
    /// @param previousValue The value before the change.
    /// @param newValue The value after the change.
    /// @param operator The address that performed the change.
    event CounterUpdated(
        uint256 previousValue,
        uint256 newValue,
        address indexed operator
    );

    /// @notice Reverts when the caller is not the owner.
    error NotOwner();

    /// @notice Restricts access to the contract owner.
    modifier onlyOwner() {
  

### Interprétation : workflow complet

Le pipeline exécute quatre preuves distinctes :

| Étape | Entrée | Sortie vérifiable |
|-------|--------|-------------------|
| **Génération** | Description en langage naturel | Réponse OpenAI et code Solidity extrait |
| **Compilation** | Code généré | Identifiant de contrat et bytecode produits par `solc` |
| **Audit** | Code compilable | Rapport de vulnérabilités issu d'un second appel OpenAI |
| **Documentation** | Code généré | Code NatSpec issu d'un troisième appel, lui aussi recompilé |

La compilation ferme un défaut important : un LLM peut produire un texte plausible mais syntaxiquement invalide. Elle ne remplace pas les tests comportementaux, qui restent l'étape suivante avec Foundry dans SC-12.

La sortie se termine par la preuve de compilation : `Contrat : <stdin>:Counter` et la taille du bytecode produite par ce run (variable d'une génération à l'autre). L'audit LLM du contrat `Counter` généré y remonte un `SEVERITY: MEDIUM` et quatre `SEVERITY: INFO`, sans CRITICAL ni HIGH : le workflow a produit un contrat compilable et un rapport d'audit structuré, en trois appels distincts au fournisseur.

## 7. LLM Local avec Qwen (Optionnel)

Pour les cas ou les API cloud ne sont pas disponibles ou souhaites, on peut utiliser un LLM local via un endpoint.

L'assistant local s'appuie sur un endpoint Ollama (`http://localhost:11434`) : le constructeur sonde `/api/tags` avec un timeout de 2 s et `available` reflète cette réponse ; si l'endpoint ne répond pas, toute génération lève `RuntimeError` au lieu de basculer silencieusement sur le cloud.

In [13]:
class LocalLLMAssistant:
    """Assistant optionnel utilisant un endpoint Ollama local."""

    def __init__(
        self,
        endpoint: str = "http://localhost:11434/api/generate",
        model: str = "qwen2.5-coder:7b",
    ):
        self.endpoint = endpoint
        self.model = model
        self.available = self._check_availability()

    def _check_availability(self) -> bool:
        """Vérifie si l'API Ollama répond."""
        try:
            response = requests.get("http://localhost:11434/api/tags", timeout=2)
            response.raise_for_status()
            return True
        except requests.RequestException:
            return False

    def generate(self, prompt: str, temperature: float = 0.2) -> str:
        """Génère une réponse via Ollama ou propage l'erreur réseau."""
        if not self.available:
            raise RuntimeError("L'endpoint Ollama optionnel n'est pas disponible.")

        response = requests.post(
            self.endpoint,
            json={
                "model": self.model,
                "prompt": prompt,
                "options": {"temperature": temperature},
                "stream": False,
            },
            timeout=90,
        )
        response.raise_for_status()
        content = response.json().get("response")
        if not content:
            raise RuntimeError("Ollama a retourné une réponse vide.")
        return content

    def generate_contract(self, description: str) -> str:
        """Génère et compile un contrat Solidity via Ollama."""
        prompt = SOLIDITY_PROMPTS["contract_generation"].format(description=description)
        code = extract_solidity_code(self.generate(prompt))
        compile_contract(code)
        return code


local_assistant = LocalLLMAssistant()
print(f"Endpoint Ollama optionnel disponible : {local_assistant.available}")
print(f"Modèle local configuré : {local_assistant.model}")

Endpoint Ollama optionnel disponible : True
Modèle local configuré : qwen2.5-coder:7b


Démonstration de l'assistant LLM local avec un modèle hébergé en local (si disponible) pour la génération de contrats.

La sortie affiche `Endpoint Ollama optionnel disponible : True` et `Modèle local configuré : qwen2.5-coder:7b` : dans ce run, le serveur Ollama local répond, donc l'extension locale est réellement disponible — sinon la cellule aurait simplement laissé `available` à `False`.

In [14]:
# Démonstration locale optionnelle, distincte du chemin principal OpenAI
if local_assistant.available:
    local_code = local_assistant.generate_contract(
        "Un contrat HelloWorld autonome qui stocke et retourne un message."
    )
    print("Contrat généré par Ollama et compilé avec solc :")
    print(local_code)
else:
    print("Extension locale non exécutée : endpoint Ollama indisponible.")
    print("Le chemin principal OpenAI a déjà fourni la preuve d'exécution de ce notebook.")

Contrat généré par Ollama et compilé avec solc :
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/**
 * @title HelloWorld
 * @author Qwen
 * @notice A simple contract that stores and retrieves a message.
 */
contract HelloWorld {
    string private message;

    /**
     * @notice Initializes the contract with an empty message.
     */
    constructor() {
        message = "";
    }

    /**
     * @notice Stores a new message.
     * @param _message The new message to store.
     */
    function store(uint256 _message) public {
        message = string(abi.encodePacked(_message));
    }

    /**
     * @notice Retrieves the stored message.
     * @return The stored message.
     */
    function retrieve() public view returns (string memory) {
        return message;
    }
}


### Lecture de la sortie : génération locale par Ollama

La sortie affiche `Contrat généré par Ollama et compilé avec solc :` puis le contrat `HelloWorld` (constructeur initialisant `message` à la chaîne vide, fonctions `store(uint256)` et `retrieve()`), produit par le modèle local `qwen2.5-coder:7b` via l'endpoint Ollama. Cette extension reste distincte du chemin principal : la preuve d'exécution du carnet est portée par les appels OpenAI des sections précédentes.

Ce que ce chemin local prouve, exactement : **la compilabilité** du contrat produit. Le contrat local n'est ni déployé ni appelé — la compilation est ici le seul contrôle déterministe appliqué. Or la compilation ne dit rien du comportement, et le contrat affiché en est la démonstration : sa fonction `store(uint256)` écrit `string(abi.encodePacked(_message))`, c'est-à-dire les **32 octets** de l'entier, complétés par des octets nuls. Un contrôle borné sur le texte exact affiché — compilation, déploiement sur PyEVM, puis lecture de `retrieve()` avant et après `store(5)` — montre le message initial remplacé par ces 32 octets : le nombre n'est pas stocké comme nombre, et la valeur relue n'est plus la chaîne initiale. Le témoin reproductible est `test_sc11_local_path.py`, à la racine de la série ; il n'exige ni serveur Ollama ni clé d'API.

C'est le principe du carnet appliqué à son propre chemin optionnel : **le LLM propose, l'outil déterministe vérifie** — et l'outil déterministe qui manque ici est l'appel de la fonction. Un contrat qui compile n'est pas un contrat qui fonctionne.

### API cloud et extension locale

| Aspect | Chemin principal OpenAI | Extension locale Ollama |
|--------|-------------------------|-------------------------|
| **Preuve dans ce carnet** | Appels réels, compilation, déploiement et test borné sur PyEVM | Compilation seule : le contrat local n'est ni déployé ni appelé |
| **Données** | Envoyées au fournisseur | Restent sur la machine locale |
| **Prérequis** | Clé API gitignored | Serveur Ollama et modèle installés |
| **Erreur** | Propagée, le notebook échoue | Propagée si une génération locale est lancée |

Cette table décrit l'architecture, pas une comparaison de qualité ou de performance. Comparer les modèles exigerait un protocole et des mesures dédiés.

## Résumé et bonnes pratiques

### Ce que nous avons appris

Le chemin principal de ce notebook suit une règle simple : **le LLM propose, les outils déterministes vérifient**. OpenAI génère le Solidity, tandis que `solc` établit sa compilabilité et que PyEVM vérifie les comportements explicitement testés. L’audit LLM et la documentation NatSpec restent des aides probabilistes qui exigent une revue humaine.

### Checklist avant déploiement

- [ ] compiler chaque contrat généré avec la version Solidity ciblée ;
- [ ] écrire des tests unitaires et des propriétés Foundry ;
- [ ] exécuter du fuzzing sur les invariants critiques ;
- [ ] lancer une analyse statique indépendante, par exemple Slither ;
- [ ] faire relire les contrôles d’accès et les interactions externes ;
- [ ] vérifier la documentation NatSpec contre le comportement réel.

**Sorties commitées à retenir**, citées telles quelles : `Entrées ABI : 6` pour `<stdin>:SimpleStorage`, avec `Lecture après store(42) : 42` et `Écriture non-propriétaire rejetée : True` ; `SEVERITY: CRITICAL` sur `withdraw()` dans l'audit de `VulnerableVault` ; `Contrat : <stdin>:Token` après recompilation NatSpec ; `Contrat : <stdin>:Counter` pour le workflow complet ; `Compilation : SUCCÈS` pour l'exemple guidé ERC-20 ; `Sévérités bloquantes finales : []` pour la boucle sécurisée. Les tailles de bytecode ne sont pas reprises : stochastiques par construction (contrat LLM différent à chaque exécution), elles re-périraient à la prochaine ré-exécution.

## Perspectives

Le pipeline complet — génération, compilation, audit et documentation — illustre une intégration industrielle minimale mais honnête. Il évite les réponses de substitution : une erreur du fournisseur, de compilation ou de test est propagée au lieu d’être transformée en faux succès.

Les exercices suivants permettent de prolonger le notebook en faisant varier les spécifications, en observant les audits et en construisant une boucle de correction bornée. Le notebook suivant, [SC-12-Foundry-Testing-Python](../03-Foundry-Testing/SC-12-Foundry-Testing-Python.ipynb), approfondit les tests Solidity.

Les exercices 1 à 4 prolongent ce pipeline sans le réexécuter : mêmes fonctions (`assistant.full_workflow()`, `audit_contract()`, `generate_natspec()`), même fournisseur, mais des contrats et des variantes à vous — sequestre, boucle journalisée, documentation comptée, comparaison d'audits.

## Exemples guidés et exercices

Les deux **exemples guidés** ci-dessous sont des resolutions d'etudiants des promotions précédentes, conservees comme demonstrations. Chaque exemple est suivi d'un **nouvel exercice** a faire vous-même. Les exercices 3 et 4 (plus bas) restent a completer.

### Exemple guidé 1 : Generer un Token ERC-20

> Contribution etudiante : **@Tooom123 (tom)** — PR #2474. Resolution conservee comme exemple guide.

Utilisation de l'assistant LLM pour generer un token ERC-20 complet (nom, symbole, decimales, fonctions mint/burn owner-only, events `Transfer`/`Approval`, documentation NatSpec), puis audit du code genere.

Chaque exemple guidé est une démonstration déjà exécutée : sa sortie (code généré, rapport d'audit, preuve de compilation) est conservée dans le carnet et citée telle quelle dans la lecture qui suit immédiatement la cellule. Les exercices proposés ensuite repartent de cette structure sans la recopier : ils demandent une variante personnelle.

In [15]:
# Exemple guidé 1 - Générer un ERC-20 avec l'assistant LLM
# Contribution originale : @Tooom123

erc20_description = """
Un token ERC-20 autonome, sans import externe, avec :
- nom MyToken, symbole MTK et 18 décimales ;
- supply initiale de 1 000 000 tokens allouée au déployeur ;
- fonctions transfer, approve et transferFrom conformes à l'interface ERC-20 ;
- fonctions mint et burn réservées au propriétaire ;
- événements Transfer et Approval ;
- documentation NatSpec complète.
"""

erc20_result = assistant.full_workflow(erc20_description)
print("Code ERC-20 généré et compilé :")
print(erc20_result["original_code"])
print("\nRapport d'audit :")
print(erc20_result["audit_report"])
print("\nCompilation : SUCCÈS")
print(f"Bytecode : {len(erc20_result['compilation']['bytecode']) // 2} octets")

Code ERC-20 généré et compilé :
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title MyToken
/// @author
/// @notice Autonomous ERC-20 token with owner-controlled minting and burning.
/// @dev This contract has no external imports or dependencies.
contract MyToken {
    /// @notice The name of the token.
    string public constant name = "MyToken";

    /// @notice The symbol of the token.
    string public constant symbol = "MTK";

    /// @notice The number of decimal places used by the token.
    uint8 public constant decimals = 18;

    /// @notice The initial token supply in base units.
    uint256 public constant INITIAL_SUPPLY = 1_000_000 * 10 ** 18;

    /// @notice The address that can mint, burn, and manage ownership.
    address public owner;

    /// @notice The total number of tokens currently in circulation.
    uint256 public totalSupply;

    /// @notice The token balance of each account.
    mapping(address => uint256) private _balances;

    /// @notic

### Lecture de la sortie : l'exemple guidé ERC-20, exécuté de bout en bout

La sortie affiche `Code ERC-20 généré et compilé :` suivi du contrat `MyToken` (name `MyToken`, symbol `MTK`, `decimals = 18`, fonctions `transfer`/`approve`/`transferFrom` et `mint`/`burn` réservées au propriétaire), puis `Rapport d'audit :`, et enfin `Compilation : SUCCÈS` avec sa taille de bytecode (propre à cette génération). L'exemple montre le pipeline complet `assistant.full_workflow()` : génération, audit et compilation réelle par `solc` sur un contrat autonome.

### Exercice 1 (a vous) : Generer un contrat de sequestre (Escrow)

En vous inspirant de l'exemple guide ci-dessus (même structure, description différente), utilisez `assistant.full_workflow()` pour generer un contrat de **sequestre (escrow)** a deux parties :
- un acheteur depose des fonds,
- le vendeur ne peut les recuperer qu'après confirmation de l'acheteur,
- l'acheteur peut se faire rembourser si la livraison n'a pas lieu.

Affichez le code genere puis le rapport d'audit.

Livrables : la description en langage naturel, le code du contrat obtenu et son rapport d'audit, affichés dans la cellule de travail. Le contrat visé doit compiler sous `solc 0.8.20` et son audit doit être lisible ligne par ligne (une sévérité par ligne `SEVERITY:`).

In [16]:
# Exercice 1 (a vous) - Espace de travail
# Etape 1 : Decrire un contrat de sequestre (escrow) a deux parties (acheteur / vendeur)
# Etape 2 : Appeler assistant.full_workflow(escrow_description)
# Etape 3 : Afficher result["original_code"] puis result["audit_report"]
# Indice : reprenez la structure de l'exemple guide 1 ci-dessus
escrow_description = """
# Votre description du contrat de sequestre (escrow) ici
"""
# result = assistant.full_workflow(escrow_description)
# print(result["original_code"])
# print(result["audit_report"])
print("Exercice a completer")

Exercice a completer


### Exemple guidé 2 : Workflow de Securite (boucle de generation auto-corrigee)

> Contribution etudiante : **@Tooom123 (tom)** — PR #2474. Resolution conservee comme exemple guide.

Une fonction qui : (1) genere un contrat a partir d'une description, (2) audite le code, (3) si des vulnerabilites CRITICAL/HIGH sont detectees, demande au LLM de corriger, puis (4) recommence jusqu'a un audit propre ou `max_iterations`. Testee ici sur un contrat de vote simple.

La démonstration suivante exécute cette boucle sur un contrat de vote : à chaque itération, les sévérités CRITICAL/HIGH extraites du rapport déclenchent une demande de correction, jusqu'à obtenir un audit sans sévérité bloquante ou atteindre `max_iterations`.

In [17]:
# Exemple guidé 2 - Boucle de génération sécurisée
# Contribution originale : @Tooom123
def blocking_severities(audit_report: str) -> set[str]:
    """Extrait les sévérités bloquantes d'un rapport au format imposé."""
    all_severities = re.findall(
        r"(?im)^\s*SEVERITY\s*:\s*(CRITICAL|HIGH|MEDIUM|LOW|INFO)\s*$",
        audit_report,
    )
    no_findings = re.search(r"(?im)^\s*NO_FINDINGS\s*$", audit_report)
    if not all_severities and not no_findings:
        raise RuntimeError(
            "Le rapport d'audit ne contient ni ligne SEVERITY structurée ni NO_FINDINGS."
        )
    return {
        severity.upper()
        for severity in all_severities
        if severity.upper() in {"CRITICAL", "HIGH"}
    }


def secure_generation_loop(description: str, max_iterations: int = 2) -> Dict[str, Any]:
    """Génère, compile, audite et corrige un contrat avec le même fournisseur."""
    code, _ = generate_contract(description)
    compilation = compile_contract(code)
    audit = audit_contract(code)

    for _ in range(max_iterations - 1):
        severities = blocking_severities(audit)
        if not severities:
            break

        correction_prompt = f"""Corrige tous les findings CRITICAL et HIGH de l'audit ci-dessous.
Retourne uniquement le contrat Solidity complet, autonome et compilable.
Conserve uniquement des caractères ASCII dans les chaînes littérales exécutées par le contrat.

Contrat :
```solidity
{code}
```

Audit :
{audit}
"""
        code = extract_solidity_code(call_llm(correction_prompt))
        compilation = compile_contract(code)
        audit = audit_contract(code)

    final_severities = sorted(blocking_severities(audit))
    return {
        "code_final": code,
        "rapport_audit": audit,
        "severites_bloquantes": final_severities,
        "converged": not final_severities,
        "compilation": compilation,
    }


voting_description = """
Un contrat de vote autonome où chaque adresse ne peut voter qu'une fois,
le propriétaire peut ajouter des candidats avant l'ouverture du scrutin,
et personne ne peut voter après la clôture.
"""

resultat = secure_generation_loop(voting_description)
print("Code final compilé :")
print(resultat["code_final"])
print("\nRapport d'audit final :")
print(resultat["rapport_audit"])
print("\nSévérités bloquantes finales :", resultat["severites_bloquantes"])
print("Compilation : SUCCÈS")
print(f"Bytecode : {len(resultat['compilation']['bytecode']) // 2} octets")
if resultat["converged"]:
    print("Verdict de boucle : aucune sévérité CRITICAL/HIGH détectée à la borne.")
else:
    print("Verdict de boucle : borne atteinte avec revue humaine requise avant utilisation.")

Code final compilé :
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

/// @title Autonomous Voting
/// @author
/// @notice Provides a self-contained voting system with one vote per address.
/// @dev The contract owner manages candidates and the voting lifecycle.
contract AutonomousVoting {
    /// @notice Represents the current lifecycle phase of the election.
    enum Phase {
        Setup,
        Open,
        Closed
    }

    /// @notice Represents a voting candidate.
    struct Candidate {
        string name;
        uint256 voteCount;
    }

    /// @notice Address with administrative permissions.
    address public owner;

    /// @notice Current phase of the election.
    Phase public phase;

    /// @notice List of all registered candidates.
    Candidate[] private candidates;

    /// @notice Indicates whether an address has already voted.
    mapping(address => bool) public hasVoted;

    /// @notice Value stored through the store function.
    uint256 private sto

### Lecture de la sortie : la boucle auto-corrigée se termine sans sévérité bloquante

La sortie affiche `Code final compilé :` (contrat `AutonomousVoting`), puis `Rapport d'audit final :`, et se termine par `Sévérités bloquantes finales : []`, `Compilation : SUCCÈS`, la taille de bytecode produite par ce run, et `Verdict de boucle : aucune sévérité CRITICAL/HIGH détectée à la borne.` : à l'issue de la boucle, `blocking_severities()` retourne un ensemble vide sur le contrat final, et ce contrat compile encore sous `solc 0.8.20`.

### Exercice 2 (à vous) : journaliser les itérations de la boucle sécurisée

À partir de l'exemple guidé 2, écrivez une variante `secure_generation_loop_journalisee` qui enregistre à chaque itération le numéro d'itération et les sévérités bloquantes retournées par `blocking_severities()`, puis retourne cet historique avec le code final.

**Indice** : stockez `sorted(blocking_severities(audit))` plutôt que de compter toutes les occurrences des mots « HIGH » et « CRITICAL » dans la prose.

Livrables : la variante journalisée, le code final, le rapport d'audit final et l'historique des itérations (numéro d'itération et sévérités bloquantes triées).

In [18]:
# Exercice 2 (à vous) - Espace de travail
# Objectif : variante de secure_generation_loop qui journalise chaque itération
# Étape 1 : repartir de l'exemple guidé 2 ci-dessus
# Étape 2 : ajouter {"iteration": i, "severites": [...]} à une liste
# Étape 3 : retourner code_final, rapport_audit et historique
# Indice : sorted(blocking_severities(audit))
def secure_generation_loop_journalisee(
    description: str, max_iterations: int = 3
) -> Dict[str, Any] | None:
    historique = []  # TODO étudiant : journal des itérations
    # TODO étudiant : implémenter la variante sans réponse préfabriquée
    return None


# test_description = "Un contrat de tirage au sort autonome"
# print(secure_generation_loop_journalisee(test_description))
print("Exercice a completer")

Exercice a completer


### Exercice 3 : Generer la documentation NatSpec d'un contrat

Reutilisez `generate_natspec()` (section 5) pour documenter automatiquement un contrat
sans commentaires. Comptez les tags `@notice` dans la sortie.

**Indice** : `generate_natspec` retourne le code documente. Comptez `"@notice"` dedans.


In [19]:
# Exercice 3 - Generer la documentation NatSpec d'un contrat
# TODO etudiant : utiliser generate_natspec() pour documenter un contrat sans commentaires
# Etape 1 : Definir contract_a_documenter (un contrat Solidity minimal sans NatSpec)
# Etape 2 : Appeler generate_natspec(contract_a_documenter)
# Etape 3 : Extraire le code et compter les tags @notice
contract_a_documenter = """
pragma solidity ^0.8.20;

contract Counter {
    uint256 public count;
    function increment() public { count += 1; }
    function reset() public { count = 0; }
}
"""
# documented = generate_natspec(contract_a_documenter)
# code = extract_solidity_code(documented)
# print(code)
# print("Tags @notice trouves :", code.count("@notice"))
print("Exercice a completer")


Exercice a completer


### Exercice 4 : comparer deux formulations d'audit

Réutilisez `audit_contract()` pour auditer le même contrat vulnérable avec deux formulations du code : une version utilisant `tx.origin`, puis une version corrigée utilisant `msg.sender`. Comparez les sévérités et les recommandations.

**Indice** : le fournisseur reste identique ; l'expérience isole l'effet de la correction du contrat plutôt qu'une différence entre deux modèles.

In [20]:
# Exercice 4 - Comparer un contrat vulnérable et sa correction
# TODO étudiant : auditer les deux versions avec le même fournisseur
# Étape 1 : conserver la version vulnérable utilisant tx.origin
# Étape 2 : écrire contrat_corrige en remplaçant ce contrôle par msg.sender
# Étape 3 : appeler audit_contract() sur chaque version et comparer les sévérités
# Indice : cherchez notamment la mention de phishing dans les rapports
contrat_vulnerable = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract PhishableWallet {
    address public owner;
    constructor() { owner = msg.sender; }

    function transferTo(address payable dest, uint256 amount) external {
        require(tx.origin == owner, "Not owner");
        dest.transfer(amount);
    }
}
"""

contrat_corrige = """
// TODO étudiant : recopier le contrat et remplacer le contrôle tx.origin
"""
# rapport_vulnerable = audit_contract(contrat_vulnerable)
# rapport_corrige = audit_contract(contrat_corrige)
# print(rapport_vulnerable)
# print(rapport_corrige)
print("Exercice a completer")

Exercice a completer


## Ressources complémentaires

### Documentation
- [OpenAI API Docs](https://platform.openai.com/docs/)
- [Ollama](https://ollama.ai/) — LLM local optionnel
- [Solidity NatSpec](https://docs.soliditylang.org/en/latest/natspec-format.html)
- [py-solc-x](https://solcx.readthedocs.io/)

### Outils
- [Foundry](https://book.getfoundry.sh/) — compilation, tests et fuzzing
- [Slither](https://github.com/crytic/slither) — analyse statique Solidity

## Conclusion

Le pipeline démontré associe une invocation OpenAI réelle à des validations locales indépendantes : génération du Solidity, compilation, déploiement et test comportemental borné sur PyEVM, audit LLM, ajout de NatSpec et recompilation. L'exemple `VulnerableVault` montre en particulier comment un rapport LLM peut guider la revue d'une réentrance et de son correctif Checks-Effects-Interactions, sans être présenté comme une analyse statique formelle.

Cette combinaison rend visibles les limites de chaque outil : le LLM propose, audite et documente de manière probabiliste ; `solc` établit que le source produit un artefact EVM valide ; le test PyEVM vérifie quelques comportements explicitement choisis. L'extension Qwen/Ollama reste optionnelle et distincte ; elle ne remplace jamais la preuve du chemin principal OpenAI.

Un test borné ne démontre pas la sûreté générale. Avant tout déploiement, complétez cette première barrière par des tests Foundry plus complets, du fuzzing, une analyse statique et une revue humaine adaptée au niveau de risque. Le notebook suivant, [SC-12-Foundry-Testing-Python](../03-Foundry-Testing/SC-12-Foundry-Testing-Python.ipynb), introduit cette étape de tests Solidity.

***

[<< Précédent : Account Abstraction](SC-10-Account-Abstraction-Python.ipynb) | [Retour au sommaire](../README.md) | [Suivant : Foundry Testing >>](../03-Foundry-Testing/SC-12-Foundry-Testing-Python.ipynb)